In [ ]:

# =========================================================
# 1. IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    RandomizedSearchCV
)

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

import warnings
warnings.filterwarnings("ignore")

In [2]:
# =========================================================
# 2. LOAD DATASET
# =========================================================

data = pd.read_csv("../Data/Energy_consumption_data.csv")



In [3]:
# =========================================================
# 5. FEATURE ENGINEERING FUNCTIONS
# =========================================================

# ---------------------------------------------------------
# Time Period Function
# ---------------------------------------------------------

def get_time_period(hour):

    if 5 <= hour < 12:
        return 0      # Morning

    elif 12 <= hour < 17:
        return 1      # Afternoon

    elif 17 <= hour < 21:
        return 2      # Evening

    else:
        return 3      # Night


# ---------------------------------------------------------
# Weekend Function
# ---------------------------------------------------------

def get_weekend_label(dayofweek):

    return 1 if dayofweek >= 5 else 0

In [4]:
# =========================================================
# 4. DATA PREPROCESSING & CLEANING
# =========================================================

# ---------------------------------------------------------
# Extract Time Features from Timestamp (Fixes KeyError: 'Hour')
# ---------------------------------------------------------
# If your column is named 'timestamp' (lowercase), change 'Timestamp' to 'timestamp' below
if 'Timestamp' in data.columns:
    data['Timestamp'] = pd.to_datetime(data['Timestamp'])
    data['Hour'] = data['Timestamp'].dt.hour
    data['Day'] = data['Timestamp'].dt.day
    data['Month'] = data['Timestamp'].dt.month
else:
    print("⚠️ Warning: 'Timestamp' column not found. Make sure your time column is parsed correctly!")

# ---------------------------------------------------------
# Convert Day Names to Numbers
# ---------------------------------------------------------
day_mapping = {
    "Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3, 
    "Friday": 4, "Saturday": 5, "Sunday": 6
}
data['DayOfWeek'] = data['DayOfWeek'].map(day_mapping)

# ---------------------------------------------------------
# Convert Holiday Column
# ---------------------------------------------------------
holiday_mapping = {"Yes": 1, "No": 0}
data['Holiday'] = data['Holiday'].map(holiday_mapping)

# ---------------------------------------------------------
# Convert HVAC & Lighting
# ---------------------------------------------------------
binary_mapping = {"On": 1, "Off": 0}
data['HVACUsage'] = data['HVACUsage'].map(binary_mapping)
data['LightingUsage'] = data['LightingUsage'].map(binary_mapping)


# =========================================================
# 5. FEATURE ENGINEERING FUNCTIONS
# =========================================================

def get_time_period(hour):
    if 5 <= hour < 12:
        return 0  # Morning
    elif 12 <= hour < 17:
        return 1  # Afternoon
    elif 17 <= hour < 21:
        return 2  # Evening
    else:
        return 3  # Night

def get_weekend_label(dayofweek):
    return 1 if dayofweek >= 5 else 0


# =========================================================
# 6. CREATE ENGINEERED FEATURES
# =========================================================
# Now 'DayOfWeek' and 'Hour' are guaranteed to exist as integers!
data['WeekendLabel'] = data['DayOfWeek'].apply(get_weekend_label)
data['TimePeriodLabel'] = data['Hour'].apply(get_time_period)

In [5]:
data


,Timestamp,Temperature,Humidity,SquareFootage,Occupancy,HVACUsage,LightingUsage,RenewableEnergy,DayOfWeek,Holiday,EnergyConsumption,Hour,Day,Month,WeekendLabel,TimePeriodLabel
0,2022-01-01 00:00:00,25.139433,43.431581,1565.693999,5,1,0,2.774699,0,0,75.364373,0,1,1,0,3
1,2022-01-01 01:00:00,27.731651,54.225919,1411.064918,1,1,1,21.831384,5,0,83.401855,1,1,1,1,3
2,2022-01-01 02:00:00,28.704277,58.907658,1755.715009,2,0,0,6.764672,6,0,78.270888,2,1,1,1,3
3,2022-01-01 03:00:00,20.080469,50.371637,1452.316318,1,0,1,8.623447,2,0,56.519850,3,1,1,0,3
4,2022-01-01 04:00:00,23.097359,51.401421,1094.130359,9,1,0,3.071969,4,0,70.811732,4,1,1,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2022-02-11 11:00:00,28.619382,48.850160,1080.087000,5,0,0,21.194696,5,0,82.306692,11,11,2,1,0
996,2022-02-11 12:00:00,23.836647,47.256435,1705.235156,4,0,1,25.748176,1,1,66.577320,12,11,2,0,1
997,2022-02-11 13:00:00,23.005340,48.720501,1320.285281,6,0,1,0.297079,4,1,72.753471,13,11,2,0,1
998,2022-02-11 14:00:00,25.138365,31.306459,1309.079719,3,1,0,20.425163,3,1,76.950389,14,11,2,0,1


In [ ]:
# =========================================================
# 7. ENHANCED FEATURE ENGINEERING
# =========================================================

# --- Cyclical encoding of Hour ---
data['HourSin'] = np.sin(2 * np.pi * data['Hour'] / 24)
data['HourCos'] = np.cos(2 * np.pi * data['Hour'] / 24)

# --- Cyclical encoding of Month ---
data['MonthSin'] = np.sin(2 * np.pi * data['Month'] / 12)
data['MonthCos'] = np.cos(2 * np.pi * data['Month'] / 12)

# --- Cyclical encoding of DayOfWeek ---
data['DayOfWeekSin'] = np.sin(2 * np.pi * data['DayOfWeek'] / 7)
data['DayOfWeekCos'] = np.cos(2 * np.pi * data['DayOfWeek'] / 7)

# --- Interaction features ---
data['Temp_x_Occupancy']  = data['Temperature'] * data['Occupancy']
data['Temp_x_HVAC']       = data['Temperature'] * data['HVACUsage']
data['Temp_squared']      = data['Temperature'] ** 2
data['Occ_x_Lighting']    = data['Occupancy']   * data['LightingUsage']
data['Occ_x_HVAC']        = data['Occupancy']   * data['HVACUsage']
data['Energy_intensity']  = data['Occupancy']   / (data['SquareFootage'] + 1)
data['Renewable_ratio']   = data['RenewableEnergy'] / (data['Temperature'] + 1)

# =========================================================
# 8. FEATURES & TARGET
# =========================================================

features = [
    # Core features
    'Temperature',
    'Humidity',
    'SquareFootage',
    'Occupancy',
    'HVACUsage',
    'LightingUsage',
    'RenewableEnergy',
    'DayOfWeek',
    'Holiday',
    'Hour',
    'Day',
    'Month',
    'WeekendLabel',
    'TimePeriodLabel',
    # Cyclical encodings
    'HourSin', 'HourCos',
    'MonthSin', 'MonthCos',
    'DayOfWeekSin', 'DayOfWeekCos',
    # Interaction features
    'Temp_x_Occupancy',
    'Temp_x_HVAC',
    'Temp_squared',
    'Occ_x_Lighting',
    'Occ_x_HVAC',
    'Energy_intensity',
    'Renewable_ratio',
]

target = 'EnergyConsumption'

X = data[features]
y = data[target]

print(f"Total features used: {len(features)}")
print(f"Dataset shape: {X.shape}")


In [ ]:
# =========================================================
# 9. TRAIN / TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# =========================================================
# 10. TRAIN GRADIENT BOOSTING MODEL
# =========================================================

model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    min_samples_leaf=3,
    min_samples_split=5,
    subsample=0.85,
    max_features='sqrt',
    random_state=42
)

model.fit(X_train, y_train)

print("Model training complete!")


In [ ]:
#save the model
import joblib
joblib.dump(model, 'energy_model.pkl')
print("Model saved as energy_model.pkl")


In [ ]:
# =========================================================
# 11. PREDICTIONS & EVALUATION
# =========================================================

y_pred = model.predict(X_test)

r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)

print("\n================ MODEL PERFORMANCE ================")
print(f"R2 Score : {r2:.4f}")
print(f"RMSE     : {rmse:.2f}")
print(f"MAE      : {mae:.2f}")

# Cross-validation score
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
print(f"\n5-Fold CV R2 : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

if r2 >= 0.80:
    print(f"\nSUCCESS: Target achieved! R2 = {r2:.4f} >= 0.80")
else:
    print(f"\nWARNING: R2 = {r2:.4f} - still below 0.80")


In [ ]:
# =========================================================
# 13. FEATURE IMPORTANCE
# =========================================================

importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
).reset_index(drop=True)

print("\n================ FEATURE IMPORTANCE ================")
print(importance_df.to_string())


In [12]:
# =========================================================
# 18. CONSUMER FRIENDLY USER INPUT
# =========================================================

user_input = {

    "PropertyType": "Home",      # Home / Commercial

    "PropertySize": "Small",    # Small / Medium / Large

    "PeopleCount": 10,

    "ACUsage": "Low",           # Low / Medium / High

    "LightingUsage": "Low",   # Low / Medium / High

    "WeatherCondition": "Hot",   # Cool / Normal / Hot

    "Hour": 20,

    "DayOfWeek": 6,

    "Holiday": 1,

    "Day": 29,

    "Month": 5

}


In [13]:
# =========================================================
# 19. BACKEND FEATURE MAPPING
# =========================================================

# ---------------------------------------------------------
# Property Size Mapping
# ---------------------------------------------------------

size_mapping = {

    "Small": 300,
    "Medium": 750,
    "Large": 1000

}

square_footage = size_mapping[
    user_input['PropertySize']
]


# ---------------------------------------------------------
# Weather Mapping
# ---------------------------------------------------------

weather_mapping = {

    "Cool": {
        "Temperature": 22,
        "Humidity": 45
    },

    "Normal": {
        "Temperature": 28,
        "Humidity": 55
    },

    "Hot": {
        "Temperature": 35,
        "Humidity": 65
    }

}

temperature = weather_mapping[
    user_input['WeatherCondition']
]['Temperature']

humidity = weather_mapping[
    user_input['WeatherCondition']
]['Humidity']


# ---------------------------------------------------------
# AC Usage Mapping
# ---------------------------------------------------------

ac_mapping = {

    "Low": 0,
    "Medium": 1,
    "High": 1

}

hvac_usage = ac_mapping[
    user_input['ACUsage']
]


# ---------------------------------------------------------
# Lighting Usage Mapping
# ---------------------------------------------------------

lighting_mapping = {

    "Low": 0,
    "Medium": 1,
    "High": 1

}

lighting_usage = lighting_mapping[
    user_input['LightingUsage']
]


# ---------------------------------------------------------
# Derived Features
# ---------------------------------------------------------

weekend_label = get_weekend_label(
    user_input['DayOfWeek']
)

time_period_label = get_time_period(
    user_input['Hour']
)


In [14]:
# =========================================================
# 20. FINAL MODEL INPUT
# =========================================================

final_input = pd.DataFrame([{

    'Temperature': temperature,

    'Humidity': humidity,

    'SquareFootage': square_footage,

    'Occupancy': user_input['PeopleCount'],

    'HVACUsage': hvac_usage,

    'LightingUsage': lighting_usage,

    'DayOfWeek': user_input['DayOfWeek'],

    'Holiday': user_input['Holiday'],

    'Hour': user_input['Hour'],

    'Day': user_input['Day'],

    'Month': user_input['Month'],

    'WeekendLabel': weekend_label,

    'TimePeriodLabel': time_period_label

}])



In [15]:
# =========================================================
# 21. ENERGY PREDICTION
# =========================================================

predicted_energy = model.predict(final_input)[0]

In [16]:

# =========================================================
# 21. DAILY ENERGY ESTIMATION
# =========================================================

energy_scaling_factor = 0.2

daily_energy_estimate = (
    predicted_energy *
    energy_scaling_factor
) 


# =========================================================
# 22. MONTHLY ENERGY ESTIMATION
# =========================================================

monthly_energy_estimate = daily_energy_estimate *30

In [17]:
# =========================================================
# 24. ELECTRICITY BILL ESTIMATION
# =========================================================

domestic_rate = 6
commercial_rate = 9


if user_input['PropertyType'] == "Home":

    electricity_rate = domestic_rate

else:

    electricity_rate = commercial_rate


daily_bill_estimate = daily_energy_estimate * electricity_rate

monthly_bill_estimate = monthly_energy_estimate * electricity_rate

In [18]:
# =========================================================
# 24. SMART RECOMMENDATION ENGINE
# =========================================================

if monthly_energy_estimate > 1200:

    recommendation = (
        "Very high energy usage detected. "
        "Reduce AC usage and optimize appliance usage."
    )

elif monthly_energy_estimate > 600:

    recommendation = (
        "Moderate energy usage detected. "
        "Monitor lighting and cooling usage."
    )

else:

    recommendation = (
        "Energy usage looks efficient."
    )

In [19]:
# =========================================================
# 25. FINAL OUTPUT
# =========================================================

print("\n================ FINAL RESULT ================")

print(f"\nEstimated Daily Consumption   : {daily_energy_estimate:.2f} kWh")

print(f"\nEstimated Monthly Consumption : {monthly_energy_estimate:.2f} kWh")

print(f"\nEstimated Daily Bill          : ₹{daily_bill_estimate:.2f}")

print(f"\nProjected Monthly Bill        : ₹{monthly_bill_estimate:.2f}")

print(f"\nRecommendation                : {recommendation}")


================ FINAL RESULT ================

Estimated Daily Consumption   : 16.91 kWh

Estimated Monthly Consumption : 507.23 kWh

Estimated Daily Bill          : ₹101.45

Projected Monthly Bill        : ₹3043.39

Recommendation                : Energy usage looks efficient.
